# Inference on random holdout sequences

In [154]:
import pandas as pd
import numpy as np
import joblib
import glob
import hashlib
from sklearn.decomposition import TruncatedSVD
import os
import re

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors

# Import all modular extractors and dictionaries
from GUI_app.bioprocessing_eda import (
    load_and_clean_data,
    compute_esm_embeddings,
    compute_georgiev_features,
    GEORGIEV_DICT, 
    compute_aac_features,
    compute_aaindex_features,
    compute_propermab_features,
    compute_ablang2_paired_embeddings,
    compute_antiberty_embeddings,
    SelfContainedTargetTransformRegressor,
    load_aaindex
)

def get_required_feature_types(features):
    required = []
    for f in features:
        f_str = str(f).upper()
        if "AAC" in f_str and 'AAC' not in required: required.append('AAC')
        if "AAINDEX" in f_str and 'AAindex' not in required: required.append('AAindex')
        if "GEORGIEV" in f_str and 'Georgiev' not in required: required.append('Georgiev')
        if "ESM" in f_str and 'ESM' not in required: required.append('ESM')
        if "SVD" in f_str and 'ESM_SVD' not in required: required.append('ESM_SVD')
        if "PROPERMAB" in f_str and 'Propermab' not in required: required.append('Propermab')
        if "ANTIBERTY" in f_str and 'AntiBERTy' not in required: required.append('AntiBERTy')
        if "ABLANG2" in f_str and 'AbLang2_Paired' not in required: required.append('AbLang2_Paired')
    return required

def get_esm_name(features):
    if any("ESM_Massive_3B_" in f for f in features): return "facebook/esm2_t36_3B_UR50D"
    if any("ESM_Big_650M_" in f for f in features): return "facebook/esm2_t33_650M_UR50D"
    if any("ESM_Large_150M_" in f for f in features): return "facebook/esm2_t30_150M_UR50D"
    if any("ESM_Medium_35M_" in f for f in features): return "facebook/esm2_t12_35M_UR50D"
    return "facebook/esm2_t6_8M_UR50D"

def get_esm_tag(features):
    if any("ESM_Massive_3B_" in f for f in features): return "ESM_Massive_3B"
    if any("ESM_Big_650M_" in f for f in features): return "ESM_Big_650M"
    if any("ESM_Large_150M_" in f for f in features): return "ESM_Large_150M"
    if any("ESM_Medium_35M_" in f for f in features): return "ESM_Medium_35M"
    if any("ESM_Small_8M_" in f for f in features): return "ESM_Small_8M"
    return "ESM"

def get_required_regions(features):
    regions = set()
    for f in features:
        f_str = str(f).upper()
        if 'VH' in f_str: regions.add('CD3_VH')
        if 'VL' in f_str: regions.add('CD3_VL')
        if 'SCFV' in f_str: regions.add('scFv')
    return list(regions) if regions else ['CD3_VH', 'CD3_VL', 'scFv']

def prepare_inference_data(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'CD3 VH_HCK': 'CD3_VH', 
        'CD3 VL_HCK': 'CD3_VL', 
        'CD3_VH_HCK': 'CD3_VH', 
        'CD3_VL_HCK': 'CD3_VL'
    })
    
    if 'CD3_VH' in df.columns and 'CD3_VL' in df.columns:
        def build_fv(r):
            vh = str(r['CD3_VH']).strip().upper() if pd.notna(r['CD3_VH']) else 'NAN'
            vl = str(r['CD3_VL']).strip().upper() if pd.notna(r['CD3_VL']) else 'NAN'
            if vh == 'NAN' or vl == 'NAN': return 'NAN'
            
            linker = ''
            if 'G4S Linker2_HCK' in df.columns and pd.notna(r['G4S Linker2_HCK']):
                val = str(r['G4S Linker2_HCK']).strip().upper()
                if val not in ['NAN', 'NONE']: linker = val
            return vh + linker + vl
            
        df['scFv'] = df.apply(build_fv, axis=1)
    return df

def fit_base_svd(df_base, region_col, esm_model_name, esm_tag, prefix, target_indices=None, base_cache=None):
    if base_cache is None: base_cache = {}
    cache_key = f"{prefix}_{esm_tag}"
    
    # 🌟 NEW: Check RAM first!
    if cache_key in base_cache:
        raw_matrix = base_cache[cache_key]
    else:
        print(f" ⚙️ Fitting SVD for {prefix} using {esm_model_name}...")
        valid_seqs = df_base[region_col].fillna("").astype(str)
        _, raw_matrix = compute_esm_embeddings(
            sequences=valid_seqs, esm_model_name=esm_model_name, 
            prefix=prefix, esm_tag=esm_tag, target_indices=target_indices
        )
        base_cache[cache_key] = raw_matrix
        
    svd = TruncatedSVD(n_components=50, random_state=42)
    svd.fit(raw_matrix)
    return svd

def extract_features_for_model(df_test, pkg, esm_name, esm_tag, ftypes, target_regions, fitted_svds, my_target_indices=None, test_cache=None):
    if test_cache is None: test_cache = {}
    df_features = pd.DataFrame(index=df_test.index)
    
    aaindex_db = None
    if 'AAindex' in ftypes:
        aaindex_db, _ = load_aaindex()
    
    for col in target_regions:
        valid_seqs = df_test[col].fillna("").astype(str)
        
        extraction_passes = [(False, f"{col}_", None)]
        if my_target_indices and col in my_target_indices:
            semantic_tag, target_indices = my_target_indices[col]
            short_hash = hashlib.md5(str(target_indices).encode('utf-8')).hexdigest()[:6]
            t_prefix = f"{col}_{semantic_tag}_{short_hash}_i-" 
            extraction_passes.append((True, t_prefix, target_indices))
            
        for is_targeted, prefix, t_idx in extraction_passes:
            
            # SNIPER LOGIC: Only extract if the loaded model explicitly requires this exact prefix!
            if not any(f.startswith(prefix) for f in pkg['features']):
                continue
                
            scope_tag = f"🎯 TARGETED ({semantic_tag})" if is_targeted else "🌍 GLOBAL"
                
            if 'ESM' in ftypes:
                cache_key = f"{prefix}ESM_{esm_tag}"
                if cache_key not in test_cache:
                    print(f" ⚙️ [{scope_tag}] Computing ESM for {col}...")
                    test_cache[cache_key] = compute_esm_embeddings(
                        sequences=valid_seqs, esm_model_name=esm_name, 
                        prefix=prefix, esm_tag=esm_tag, target_indices=t_idx
                    )
                
                esm_dict, raw_matrix = test_cache[cache_key]
                
                if 'ESM_SVD' in ftypes:
                    svd_transformed = fitted_svds[prefix].transform(raw_matrix)
                    for i in range(svd_transformed.shape[1]):
                        df_features[f"{prefix}{esm_tag}_SVD50_{i}"] = svd_transformed[:, i]
                else:
                    df_features = pd.concat([df_features, pd.DataFrame(esm_dict, index=df_test.index)], axis=1)
                    
            if 'Georgiev' in ftypes:
                cache_key = f"{prefix}Georgiev"
                if cache_key not in test_cache:
                    print(f" ⚙️ [{scope_tag}] Computing Georgiev for {col}...")
                    test_cache[cache_key] = compute_georgiev_features(valid_seqs, GEORGIEV_DICT, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(test_cache[cache_key], index=df_test.index)], axis=1)

            if 'AAC' in ftypes:
                cache_key = f"{prefix}AAC"
                if cache_key not in test_cache:
                    print(f" ⚙️ [{scope_tag}] Computing AAC for {col}...")
                    test_cache[cache_key] = compute_aac_features(valid_seqs, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(test_cache[cache_key], index=df_test.index)], axis=1)
                
            if 'AAindex' in ftypes:
                cache_key = f"{prefix}AAindex"
                if cache_key not in test_cache:
                    print(f" ⚙️ [{scope_tag}] Computing AAindex for {col}...")
                    test_cache[cache_key] = compute_aaindex_features(valid_seqs, aaindex_db, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(test_cache[cache_key], index=df_test.index)], axis=1)
                
            if 'AntiBERTy' in ftypes:
                cache_key = f"{prefix}AntiBERTy"
                if cache_key not in test_cache:
                    print(f" ⚙️ [{scope_tag}] Computing AntiBERTy for {col}...")
                    test_cache[cache_key] = compute_antiberty_embeddings(valid_seqs.tolist(), prefix=prefix)
                df_features = pd.concat([df_features, pd.DataFrame(test_cache[cache_key], index=df_test.index)], axis=1)

    # ==========================================
    # 🌟 NEW: Paired & 3D Structural Features with Disk Caching
    # ==========================================
    if 'Propermab' in ftypes:
        cache_key = "Propermab_CD3"
        if cache_key not in test_cache:
            h_seqs = df_test['CD3_VH'].fillna("").astype(str).tolist()
            l_seqs = df_test['CD3_VL'].fillna("").astype(str).tolist()
            
            # Create an inference cache folder and generate a unique hash for this dataset
            os.makedirs("inference_cache", exist_ok=True)
            seq_hash = hashlib.md5("".join(h_seqs + l_seqs).encode('utf-8')).hexdigest()[:8]
            disk_cache_file = f"inference_cache/Propermab_CD3_{seq_hash}.npz"
            
            if os.path.exists(disk_cache_file):
                print(f" ⚙️ [🌍 PAIRED] Loaded Propermab 3D Physics from disk cache...")
                with np.load(disk_cache_file, allow_pickle=True) as data:
                    test_cache[cache_key] = {k: data[k] for k in data.files}
            else:
                print(" ⚙️ [🌍 PAIRED] Computing Propermab 3D Physics (This takes ~30s per seq)...")
                propermab_dict = compute_propermab_features(h_seqs, l_seqs, prefix="Propermab_CD3_")
                test_cache[cache_key] = propermab_dict
                np.savez(disk_cache_file, **propermab_dict)
                print(f" 💾 Saved Propermab features to disk cache: {disk_cache_file}")
                
        df_features = pd.concat([df_features, pd.DataFrame(test_cache[cache_key], index=df_test.index)], axis=1)

    if 'AbLang2_Paired' in ftypes:
        cache_key = "AbLang2_Paired"
        if cache_key not in test_cache:
            h_seqs = df_test['CD3_VH'].fillna("").astype(str).tolist()
            l_seqs = df_test['CD3_VL'].fillna("").astype(str).tolist()
            
            os.makedirs("inference_cache", exist_ok=True)
            seq_hash = hashlib.md5("".join(h_seqs + l_seqs).encode('utf-8')).hexdigest()[:8]
            disk_cache_file = f"inference_cache/AbLang2_Paired_{seq_hash}.npz"
            
            if os.path.exists(disk_cache_file):
                print(f" ⚙️ [🌍 PAIRED] Loaded AbLang2 from disk cache...")
                with np.load(disk_cache_file, allow_pickle=True) as data:
                    test_cache[cache_key] = {k: data[k] for k in data.files}
            else:
                print(" ⚙️ [🌍 PAIRED] Computing AbLang2 Paired Embeddings...")
                ablang_dict = compute_ablang2_paired_embeddings(h_seqs, l_seqs, prefix="Paired_CD3_VH_VL_AbLang2_")
                test_cache[cache_key] = ablang_dict
                np.savez(disk_cache_file, **ablang_dict)
                print(f" 💾 Saved AbLang2 features to disk cache: {disk_cache_file}")
                
        df_features = pd.concat([df_features, pd.DataFrame(test_cache[cache_key], index=df_test.index)], axis=1)

    df_features = df_features.loc[:, ~df_features.columns.duplicated()]

    # TABULAR FEATURE SWEEPER
    for required_feature in pkg['features']:
        if required_feature not in df_features.columns:
            if required_feature in df_test.columns:
                df_features[required_feature] = df_test[required_feature].values
            else:
                print(f" ⚠️ CRITICAL WARNING: Model requires '{required_feature}', but it is missing!")

    return df_features

def evaluate_and_save_excel(df_test, predictions, actual_col_keyword, output_excel, target_name, percent_range):
    """
    Saves the multi-sheet Excel for a single model and returns the metrics for the combined plot.
    """
    print(f"\n--- Evaluating {target_name} Performance ---")
    
    seq_ids = df_test['ID'] if 'ID' in df_test.columns else (df_test['Samples'] if 'Samples' in df_test.columns else df_test.index)
    base_results_df = pd.DataFrame({
        'Sequence_ID': seq_ids,
        f'Predicted_{target_name}': predictions
    })
    
    actual_col = next((c for c in df_test.columns if actual_col_keyword.lower() in c.lower()), None)
    
    if not actual_col:
        print(f"⚠️ Actual column for '{actual_col_keyword}' not found. Skipping evaluation.")
        return [], [], [], [], 0.0
        
    base_results_df[f'Actual_{target_name}'] = df_test[actual_col]
    
    base_results_df['Actual_Rank'] = base_results_df[f'Actual_{target_name}'].rank(method='min')
    base_results_df['Predicted_Rank'] = base_results_df[f'Predicted_{target_name}'].rank(method='min')
    
    overall_spearman = base_results_df[f'Actual_{target_name}'].corr(base_results_df[f'Predicted_{target_name}'], method='spearman')
    print(f"Overall Spearman Rank Correlation on Unseen Data: {overall_spearman:.3f}")
    
    summary_data = []
    hit_rates_for_plot = []
    abs_hits_for_plot = []  # Track absolute hits
    top_k_for_plot = []     # Track the 'N' threshold sizes
    thresholds_for_plot = [int(p * 100) for p in percent_range]

    # Write multi-sheet Excel
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        for p in percent_range:
            top_k = max(1, int(len(base_results_df) * p))
            
            sheet_df = base_results_df.copy()
            sheet_df['Is_Hit'] = (sheet_df['Actual_Rank'] <= top_k) & (sheet_df['Predicted_Rank'] <= top_k)
            
            hits = sheet_df['Is_Hit'].sum()
            hit_percentage = (hits / top_k) * 100 if top_k > 0 else 0
            
            hit_rates_for_plot.append(hit_percentage)
            abs_hits_for_plot.append(hits)
            top_k_for_plot.append(top_k)
            
            summary_data.append({
                'Top Tier Target': f"Top {int(p*100)}% (N={top_k})",
                'Hit Rate (Fraction & %%)': f"{hits}/{top_k} ({hit_percentage:.1f}%)"
            })
            
            sheet_name = f"Top_{int(p*100)}_Percent"
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
        summary_df = pd.DataFrame(summary_data)
        summary_df.loc[len(summary_df)] = ["", ""]
        summary_df.loc[len(summary_df)] = ["Overall Spearman Correlation", f"{overall_spearman:.3f}"]
        
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        workbook = writer.book
        summary_sheet = workbook['Summary']
        workbook._sheets.remove(summary_sheet)
        workbook._sheets.insert(0, summary_sheet)

    print(f"🚀 Excel results saved to {output_excel}")
    # CHANGED: Now returning absolute hits and top_k as well
    return thresholds_for_plot, hit_rates_for_plot, abs_hits_for_plot, top_k_for_plot, overall_spearman

def plot_combined_model_comparison(target_name, results_dict, thresholds, top_k_counts, output_filename):
    """
    Generates a master visual dashboard comparing unseen inference performance.
    Groups models logically (by region, then base feature, then complexity) like an ablation study.
    Colors the y-axis text based on Region + Base Feature to perfectly highlight ablation trails.
    """
    if not results_dict:
        return
        
    sns.set_theme(style="whitegrid")
    
    # Use gridspec to make the heatmap slightly wider to accommodate the y-axis labels
    fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [1.3, 1]})
    fig.suptitle(f'Unseen Data Inference Comparison: {target_name}', fontsize=18, fontweight='bold', y=1.02)
    
    # =========================================================
    # Ablation-Style Logical Sorting
    # =========================================================
    def ablation_sort_key(item):
        name = item[0]
        if ' | ' in name:
            regions_part, feats_part = name.split(' | ', 1)
        else:
            regions_part, feats_part = name, ""
            
        # 1. Group by Region (VH -> VL -> VH+VL -> scFv)
        region_rank = 99
        if regions_part == 'VH': region_rank = 1
        elif regions_part == 'VL': region_rank = 2
        elif regions_part == 'VH+VL': region_rank = 3
        elif 'scFv' in regions_part: region_rank = 4
        
        feats = feats_part.split('+') if feats_part else []
        
        # 2. Group by Base Feature to keep ablation paths perfectly stacked!
        base_feat = feats[0].strip() if feats else ""
        
        # 3. Sort by Feature Complexity within the ablation path
        num_feats = len(feats)
        num_targeted = sum(1 for f in feats if 'i-' in f)
        num_tabular = sum(1 for f in feats if any(k in f for k in ['dG', 'Kd', 'Titer']))
        
        # 4. Tie-breaker is alphabetical
        return (region_rank, base_feat, num_feats, num_targeted, num_tabular, feats_part)

    # Apply the logical sorting
    sorted_items = sorted(results_dict.items(), key=ablation_sort_key)
    # =========================================================
    
    model_names = [item[0] for item in sorted_items]
    spearmans = [item[1]['spearman'] for item in sorted_items]
    
    # Extract hit rates into a 2D matrix
    hit_rate_matrix = np.array([item[1]['hit_rates'] for item in sorted_items])
    
    # Provide ONLY the percentages to Seaborn initially
    annot_matrix = np.empty((len(model_names), len(thresholds)), dtype=object)
    for i, item in enumerate(sorted_items):
        for j in range(len(thresholds)):
            pct = item[1]['hit_rates'][j]
            annot_matrix[i, j] = f"{pct:.1f}%"
    
    # --- PANEL 1: Hit Rate Heatmap ---
    sns.heatmap(hit_rate_matrix, annot=annot_matrix, fmt="", cmap="Blues", 
                cbar_kws={'label': 'True Hits Found (%)'}, ax=axes[0],
                linewidths=1, linecolor='white', annot_kws={'fontsize': 12})
                
    # # Manually inject the absolute counts with a smaller font
    # for i, item in enumerate(sorted_items):
    #     for j in range(len(thresholds)):
    #         hits = item[1]['abs_hits'][j]
    #         top_k = top_k_counts[j]
            
    #         text_idx = i * len(thresholds) + j
    #         base_text = axes[0].texts[text_idx]
    #         text_color = base_text.get_color()
            
    #         x, y = base_text.get_position()
    #         base_text.set_position((x, y - 0.12))
            
    #         axes[0].text(x, y + 0.18, f"({hits}/{top_k})", 
    #                      ha='center', va='center', color=text_color, 
    #                      fontsize=6, alpha=0.85)
    
    # Format Heatmap Axes
    axes[0].set_title('Top-Tier Hit Rate Across Thresholds', fontsize=14, pad=15)
    axes[0].set_xlabel('Top Tier Threshold Evaluated (Threshold % and Total N)', fontsize=12, labelpad=10)
    axes[0].set_xticks(np.arange(len(thresholds)) + 0.5)
    axes[0].set_xticklabels([f"{t}%\n(N={k})" for t, k in zip(thresholds, top_k_counts)], fontsize=14)
    
    axes[0].set_yticks(np.arange(len(model_names)) + 0.5)
    axes[0].set_yticklabels(model_names, rotation=0, fontsize=11)
    
    # =========================================================
    # Color code the y-axis labels based on Region + Base Feature
    # =========================================================
    unique_trails = []
    for name in model_names:
        if ' | ' in name:
            region_part, feats_part = name.split(' | ', 1)
        else:
            region_part, feats_part = name, ""
            
        base_feat = feats_part.split('+')[0].strip() if feats_part else "None"
        
        # We combine Region AND Base Feature to make a truly unique identifier
        trail_key = f"{region_part} | {base_feat}"
        
        if trail_key not in unique_trails:
            unique_trails.append(trail_key)
            
    # Generate a dark, highly readable palette for the text based on the number of unique trails
    palette = sns.color_palette("dark", len(unique_trails)).as_hex()
    trail_colors = dict(zip(unique_trails, palette))
    
    # Apply the mapped colors
    for label in axes[0].get_yticklabels():
        text = label.get_text()
        if ' | ' in text:
            region_part, feats_part = text.split(' | ', 1)
        else:
            region_part, feats_part = text, ""
            
        base_feat = feats_part.split('+')[0].strip() if feats_part else "None"
        trail_key = f"{region_part} | {base_feat}"
        
        color = trail_colors.get(trail_key, 'black')
        label.set_color(color)
        label.set_fontweight('bold')
    # =========================================================

    # --- PANEL 2: Spearman Leaderboard (Bar Chart) ---
    
    # Dynamic coloring based on Spearman score
    min_sp, max_sp = min(spearmans), max(spearmans)
    norm = mcolors.Normalize(vmin=min_sp - (max_sp - min_sp) * 0.2, vmax=max_sp)
    cmap = plt.get_cmap('Greens')
    bar_colors = [cmap(norm(sp)) for sp in spearmans]
    
    y_positions = np.arange(len(model_names)) + 0.5
    axes[1].barh(y_positions, spearmans, height=0.6, color=bar_colors, edgecolor='black', alpha=0.9)
    axes[1].set_ylim(axes[0].get_ylim())
    
    axes[1].set_title('Out-of-Sample Spearman Rank Correlation', fontsize=14, pad=15)
    axes[1].set_xlabel('Spearman Correlation Score', fontsize=12, labelpad=10)
    axes[1].set_xlim(0, max(max(spearmans) + 0.1, 0.75))
    
    axes[1].set_yticks(y_positions)
    axes[1].set_yticklabels([]) 
    
    for i, v in enumerate(spearmans):
        axes[1].text(v + 0.01, y_positions[i], f"{v:.3f}", va='center', fontweight='bold', fontsize=11)
        
    plt.tight_layout()
    plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"\n📊 Master Dashboard successfully saved to: {output_filename}")

def get_clean_model_name(basename):
    """
    Parses a long joblib filename and converts it into a clean, compact abbreviation.
    Uses regex and masking to safely isolate structural regions from features.
    """
    # 1. Isolate the core string by removing the XGBoost/SVR prefix
    tag = basename
    if 'CD3_' in tag:
        tag = 'CD3_' + tag.split('CD3_', 1)[1]
    elif 'scFv' in tag:
        tag = 'scFv' + tag.split('scFv', 1)[1]
        
    # 2. Safely separate structural regions from features using regex
    match = re.match(r'^((?:CD3_VH|CD3_VL|scFv)(?:-(?:CD3_VH|CD3_VL|scFv))*?)_(.*)$', tag)
    
    if match:
        subregions = match.group(1)
        features = match.group(2)
    else:
        subregions = tag
        features = ""
        
    # 3. Clean up the structural subregions
    subregions = subregions.replace("CD3_VH", "VH").replace("CD3_VL", "VL").replace("-", "+")
    
    if features:
        # 4. Protect features with internal hyphens FIRST
        features = features.replace("50-50_HCCF_Titer", "Titer")
        features = features.replace("i-", "INTERFACE_PROTECT_")
        
        # 5. Now it is safe to swap the feature delimiter from hyphen to plus
        features = features.replace("-", "+")
        
        # 6. Restore the protected 'i-' prefix
        features = features.replace("INTERFACE_PROTECT_", "i-")
        
        # 7. Apply all remaining abbreviations safely
        mapping = {
            "Targeted_": "i-", # Kept just in case you ever load an older model
            "Georgiev": "Geo",
            "AAindex": "AAidx",
            "ESM_Big_650M_SVD50": "ESM(SVD)",
            "ESM_Big_650M": "ESM650",
            "ESM_Medium_35M": "ESM35",
            "ESM_Small_8M": "ESM8",
            "AntiBERTy": "ABerty",
            "AbLang2_Paired": "AbL2",
            "Propermab": "pm3D",
            "Delta_G_Rank1": "dG",
            "VH_VL_Log10_Kd": "Kd"
        }
        for old, new in mapping.items():
            features = features.replace(old, new)
            
        short_name = f"{subregions} | {features}"
    else:
        short_name = subregions
        
    # Final safety truncate just in case it's still too long for the chart
    if len(short_name) > 40:
        short_name = short_name[:37] + "..."
        
    return short_name

def main():
    base_csv_poly = "data/tubespin_subset.csv" 
    base_csv_hmw = "data/50-50_sequences_subset.csv" 
    
    percent_range = [0.20, 0.30, 0.40, 0.50, 0.60]
    
    my_target_indices = {
        'CD3_VH': ('Interface_looseness', [34, 36, 38, 42, 43, 44, 45, 46, 49, 60, 61, 62, 63, 96, 101, 102, 103, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117]),
        'CD3_VL': ('Interface_looseness', [30, 33, 34, 35, 36, 37, 39, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55, 56, 57, 88, 90, 92, 94, 95, 96, 97, 98, 99, 100, 101]),
        'scFv': ('Interface_looseness', [# Original VH indices
                                            34, 36, 38, 42, 43, 44, 45, 46, 49, 60, 61, 62, 63, 96, 101, 102, 103, 107, 108,
                                            109, 110, 111, 112, 113, 114, 115, 116, 117,
                                            # Shifted VL indices (+140)
                                            170, 173, 174, 175, 176, 177, 179, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 194,
                                            195, 196, 197, 228, 230, 232, 234, 235, 236, 237, 238, 239, 240, 241])
    }
    
    # ==========================================
    # 1. POLYREACTIVITY PIPELINE
    # ==========================================
    poly_model_paths = glob.glob("trained_models/*ELISA_Polyreactivity_Excell*.joblib")
    
    if poly_model_paths:
        print(f"\n🔹 Found {len(poly_model_paths)} Polyreactivity models! Running batch inference...")
        poly_holdout_csv = "data/tubespin_unseen.csv" 
        df_base_poly = prepare_inference_data(base_csv_poly)
        df_test_poly = prepare_inference_data(poly_holdout_csv)
        poly_results = {}
        poly_thresholds = []
        
        # Initialize RAM caches OUTSIDE the loop!
        poly_base_cache = {}
        poly_test_cache = {}
        poly_fitted_svds = {}
        
        for path in poly_model_paths:
            basename = os.path.splitext(os.path.basename(path))[0]
            output_excel = f"model_comparison/{basename}_Predictions.xlsx"
            
            clean_name = get_clean_model_name(basename)
            pkg = joblib.load(path)
            ftypes = get_required_feature_types(pkg['features'])
            esm_name = get_esm_name(pkg['features'])
            esm_tag = get_esm_tag(pkg['features'])
            regions = get_required_regions(pkg['features'])
            
            if 'ESM_SVD' in ftypes:
                for reg in regions:
                    # CHANGED: Pass the caches and check the global poly_fitted_svds dict
                    if f"{reg}_" not in poly_fitted_svds:
                        poly_fitted_svds[f"{reg}_"] = fit_base_svd(df_base_poly, reg, esm_name, esm_tag, prefix=f"{reg}_", base_cache=poly_base_cache)
                        
                    if reg in my_target_indices:
                        sem_tag, t_idx = my_target_indices[reg]
                        short_hash = hashlib.md5(str(t_idx).encode('utf-8')).hexdigest()[:6]
                        t_prefix = f"{reg}_{sem_tag}_{short_hash}_i-"
                        
                        if t_prefix not in poly_fitted_svds:
                            poly_fitted_svds[t_prefix] = fit_base_svd(df_base_poly, reg, esm_name, esm_tag, prefix=t_prefix, target_indices=t_idx, base_cache=poly_base_cache)

            # CHANGED: Pass the caches
            X = extract_features_for_model(df_test_poly, pkg, esm_name, esm_tag, ftypes, regions, poly_fitted_svds, my_target_indices, test_cache=poly_test_cache)
            pred = pkg['model'].predict(X[pkg['features']]).flatten()
            
            poly_thresholds, hit_rates, abs_hits, top_ks, spearman = evaluate_and_save_excel(df_test_poly, pred, 'Poly', output_excel, 'Poly', percent_range)
            poly_results[clean_name] = {'hit_rates': hit_rates, 'abs_hits': abs_hits, 'spearman': spearman}
            poly_top_ks = top_ks
            
        plot_combined_model_comparison('Polyreactivity', poly_results, poly_thresholds, poly_top_ks, "model_comparison/Combined_Polyreactivity_Comparison.png")


    # ==========================================
    # 2. HMW PIPELINE
    # ==========================================
    hmw_model_paths = glob.glob("trained_models/*HMW*.joblib")
    
    if hmw_model_paths:
        print(f"\n🔹 Found {len(hmw_model_paths)} HMW models! Running batch inference...")
        hmw_holdout_csv = "data/50-50_sequences_unseen.csv" 
        df_base_hmw = prepare_inference_data(base_csv_hmw)
        df_test_hmw = prepare_inference_data(hmw_holdout_csv)
        hmw_results = {}
        hmw_thresholds = []
        
        # Initialize RAM caches OUTSIDE the loop!
        hmw_base_cache = {}
        hmw_test_cache = {}
        hmw_fitted_svds = {}
        
        for path in hmw_model_paths:
            basename = os.path.splitext(os.path.basename(path))[0]
            output_excel = f"model_comparison/{basename}_Predictions.xlsx"
            
            clean_name = get_clean_model_name(basename)
            pkg = joblib.load(path)
            ftypes = get_required_feature_types(pkg['features'])
            esm_name = get_esm_name(pkg['features'])
            esm_tag = get_esm_tag(pkg['features'])
            regions = get_required_regions(pkg['features'])
            
            if 'ESM_SVD' in ftypes:
                for reg in regions:
                    if f"{reg}_" not in hmw_fitted_svds:
                        hmw_fitted_svds[f"{reg}_"] = fit_base_svd(df_base_hmw, reg, esm_name, esm_tag, prefix=f"{reg}_", base_cache=hmw_base_cache)
                        
                    if reg in my_target_indices:
                        sem_tag, t_idx = my_target_indices[reg]
                        short_hash = hashlib.md5(str(t_idx).encode('utf-8')).hexdigest()[:6]
                        t_prefix = f"{reg}_{sem_tag}_{short_hash}_i-"
                        
                        if t_prefix not in hmw_fitted_svds:
                            hmw_fitted_svds[t_prefix] = fit_base_svd(df_base_hmw, reg, esm_name, esm_tag, prefix=t_prefix, target_indices=t_idx, base_cache=hmw_base_cache)

            X = extract_features_for_model(df_test_hmw, pkg, esm_name, esm_tag, ftypes, regions, hmw_fitted_svds, my_target_indices, test_cache=hmw_test_cache)
            pred = pkg['model'].predict(X[pkg['features']]).flatten()
            
            hmw_thresholds, hit_rates, abs_hits, top_ks, spearman = evaluate_and_save_excel(df_test_hmw, pred, 'HMW', output_excel, 'HMW', percent_range)
            hmw_results[clean_name] = {'hit_rates': hit_rates, 'abs_hits': abs_hits, 'spearman': spearman}
            hmw_top_ks = top_ks
            
        plot_combined_model_comparison('HMW', hmw_results, hmw_thresholds, hmw_top_ks, "model_comparison/Combined_HMW_Comparison_Best_Performers_Extended.png")

if __name__ == "__main__":
    main()


🔹 Found 25 HMW models! Running batch inference...
 ⚙️ [🌍 GLOBAL] Computing ESM for CD3_VL...
 ⚙️ [🌍 GLOBAL] Computing AAC for CD3_VL...
 ⚙️ [🌍 GLOBAL] Computing AAindex for CD3_VL...
 ⚙️ [🎯 TARGETED (Interface_looseness)] Computing ESM for CD3_VL...
 ⚙️ [🎯 TARGETED (Interface_looseness)] Computing AAC for CD3_VL...
 ⚙️ [🎯 TARGETED (Interface_looseness)] Computing AAindex for CD3_VL...
 ⚙️ [🌍 GLOBAL] Computing ESM for CD3_VH...
 ⚙️ [🌍 GLOBAL] Computing AAC for CD3_VH...
 ⚙️ [🌍 GLOBAL] Computing AAindex for CD3_VH...
 ⚙️ [🎯 TARGETED (Interface_looseness)] Computing ESM for CD3_VH...
 ⚙️ [🎯 TARGETED (Interface_looseness)] Computing AAC for CD3_VH...
 ⚙️ [🎯 TARGETED (Interface_looseness)] Computing AAindex for CD3_VH...

--- Evaluating HMW Performance ---
Overall Spearman Rank Correlation on Unseen Data: 0.334
🚀 Excel results saved to model_comparison/Production_targeted_XGBoost_SUBSET_50-50_HMW%_CD3_VH-CD3_VL_i-AAC-i-AAindex-i-ESM_Big_650M-VH_VL_Log10_Kd_Predictions.xlsx
 ⚙️ [🌍 GLOBAL] C

# Generate Dashboard with OOD performance based on Mutation Splits 

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

def generate_detailed_ood_dashboard(csv_path, output_png="OOD_Detailed_Dashboard.png"):
    print(f"\n🌍 Loading OOD Leaderboard: {csv_path}")
    df = pd.read_csv(csv_path)
    
    # ==========================================
    # Data Formatting
    # ==========================================
    # 1. Clean the regions just like the Feature Impact script
    if 'Subregions' in df.columns:
        df['Region'] = df['Subregions'].replace({
            'CD3_VH + CD3_VL': 'VH+VL', 'CD3_VH': 'VH', 'CD3_VL': 'VL',
            'CD3_VH + scFv': 'VH+scFv', 'CD3_VL + scFv': 'VL+scFv'
        }).astype(str).str.replace(' ', '')
    else:
        df['Region'] = 'Unknown'

    region_order = ['scFv', 'VH+VL', 'VL', 'VH']
    existing_regions = [r for r in region_order if r in df['Region'].unique()]
    
    # 2. Identify the exact Split columns
    split_cols = [c for c in df.columns if c.endswith('_Spearman') and c != 'CV_Spearman']
    df = df.dropna(subset=split_cols, how='all').copy()

    # 3. Create a melted dataframe specifically for the Violin plots
    melt_cols = ['CV_Spearman'] + split_cols
    df_melted = df.melt(id_vars=['Rank', 'Features', 'Num_Dims', 'Region'], value_vars=melt_cols, 
                        var_name='Evaluation_Type', value_name='Spearman_Score')
    # Clean string names for the x-axis
    df_melted['Evaluation_Type'] = df_melted['Evaluation_Type'].str.replace('_Spearman', '').str.replace('Split_', '')

    # ==========================================
    # Figure Setup (The 4x3x3 Grid)
    # ==========================================
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(28, 22))
    fig.suptitle("High-Resolution Out-of-Distribution (OOD) Diagnostics", fontsize=28, fontweight='bold', y=0.96)
    
    # We use 12 columns to perfectly balance rows of 4 and rows of 3
    gs = gridspec.GridSpec(3, 12, figure=fig, hspace=0.35, wspace=0.5)

    # ==========================================
    # ROW 1: Violins by Region (4 plots)
    # ==========================================
    # Lock the y-axis so all 4 regions can be compared fairly side-by-side
    global_min = df_melted['Spearman_Score'].min() - 0.1
    global_max = df_melted['Spearman_Score'].max() + 0.1
    
    for idx, reg in enumerate(existing_regions[:4]): # Ensure max 4
        # Each plot takes exactly 3 columns (12 / 4 = 3)
        ax = fig.add_subplot(gs[0, idx*3:(idx+1)*3])
        subset = df_melted[df_melted['Region'] == reg]
        
        if subset.empty:
            continue
            
        sns.violinplot(data=subset, x='Evaluation_Type', y='Spearman_Score', ax=ax, 
                       hue='Evaluation_Type', legend=False,
                       palette="muted", inner="quartile", density_norm="width", cut=0)
        sns.stripplot(data=subset, x='Evaluation_Type', y='Spearman_Score', ax=ax, 
                      color="black", alpha=0.15, size=3, jitter=True)
        
        ax.set_title(f"Distribution: {reg}", fontsize=18, fontweight='bold')
        ax.set_ylabel("Spearman Correlation" if idx == 0 else "", fontsize=14)
        ax.set_xlabel("")
        ax.tick_params(axis='x', rotation=15, labelsize=11)
        ax.set_ylim([global_min, global_max]) 

    # ==========================================
    # ROW 2: Generalization Gap vs Complexity (3 plots)
    # ==========================================
    plot_splits = split_cols[:3]
    
    for idx, split_col in enumerate(plot_splits):
        # Each plot takes exactly 4 columns (12 / 3 = 4)
        ax = fig.add_subplot(gs[1, idx*4:(idx+1)*4])
        clean_name = split_col.replace('_Spearman', '').replace('Split_', '')
        
        # Calculate the exact gap for THIS specific split
        gap = df['CV_Spearman'] - df[split_col]
        
        scatter = ax.scatter(df['Num_Dims'], gap, c=df['CV_Spearman'], 
                             cmap='viridis', alpha=0.7, edgecolors='k', s=70)
        
        # Plot the Trendline safely
        valid_mask = ~gap.isna()
        if valid_mask.sum() > 1:
            z = np.polyfit(df.loc[valid_mask, 'Num_Dims'], gap[valid_mask], 1)
            p = np.poly1d(z)
            x_range = np.linspace(df['Num_Dims'].min(), df['Num_Dims'].max(), 100)
            ax.plot(x_range, p(x_range), "r--", linewidth=2.5, label=f"Overfit Trendline")

        ax.set_title(f"Overfitting Trend: {clean_name}", fontsize=18, fontweight='bold')
        ax.set_xlabel("Model Dimensionality (Num_Dims)", fontsize=14)
        ax.set_ylabel("Gap (CV - OOD Score)" if idx == 0 else "", fontsize=14)
        
        # Only put the colorbar on the far-right plot to save horizontal space
        if idx == 2:
            cbar = fig.colorbar(scatter, ax=ax)
            cbar.set_label("Original CV Spearman", fontsize=12)
            
        ax.legend(loc="upper left", fontsize=12)

    # ==========================================
    # ROW 3: Direct CV vs OOD Scatter Plots
    # ==========================================
    for idx, split_col in enumerate(plot_splits):
        # Each plot takes exactly 4 columns (12 / 3 = 4)
        ax = fig.add_subplot(gs[2, idx*4:(idx+1)*4])
        clean_name = split_col.replace('_Spearman', '').replace('Split_', '')
        
        scatter = ax.scatter(df['CV_Spearman'], df[split_col], c=df['Num_Dims'], 
                             cmap='coolwarm', alpha=0.8, edgecolors='k', s=80)
        
        min_val = min(df['CV_Spearman'].min(), df[split_col].min()) - 0.05
        max_val = max(df['CV_Spearman'].max(), df[split_col].max()) + 0.05
        
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='Perfect Generalization')
        ax.fill_between([min_val, max_val], [min_val, max_val], [min_val, min_val], color='red', alpha=0.05, label='Overfitting Zone')
        
        ax.set_title(f"Performance: {clean_name}", fontsize=18, fontweight='bold')
        ax.set_xlabel("Original CV Spearman", fontsize=14)
        ax.set_ylabel("OOD Test Spearman" if idx == 0 else "", fontsize=14)
        ax.set_xlim([min_val, max_val])
        ax.set_ylim([min_val, max_val])
        
        # Only put the colorbar on the far-right plot
        if idx == 2:
            cbar = fig.colorbar(scatter, ax=ax)
            cbar.set_label("Model Dimensionality (Num_Dims)", fontsize=12)
            
        ax.legend(loc="upper left", fontsize=12)

    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"📊 4x3x3 Dashboard successfully saved to: {output_png}")

# Execute:
# generate_detailed_ood_dashboard('OOD_Robustness_Leaderboard_XGBoost_SUBSET_50-50_HMW%.csv', 'All_Champions_OOD_Detailed.png')

In [28]:
# leaderboard_csv = 'model_comparison/OOD_Robustness_Leaderboard_XGBoost_SUBSET_50-50_HMW%.csv'
# generate_detailed_ood_dashboard(leaderboard_csv, "model_comparison/All_Champions_OOD_Dashboard_SUBSET.png")
# leaderboard_csv = 'model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%.csv'
# generate_detailed_ood_dashboard(leaderboard_csv, "model_comparison/All_Champions_OOD_Dashboard_FULLSET.png")
leaderboard_csv = 'model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer.csv'
generate_detailed_ood_dashboard(leaderboard_csv, "model_comparison/All_Champions_OOD_Dashboard_FULLSET_without_Titer.png")


🌍 Loading OOD Leaderboard: model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer.csv
📊 4x3x3 Dashboard successfully saved to: model_comparison/All_Champions_OOD_Dashboard_FULLSET_without_Titer.png


# Filter for the best models based on mutation splits generalizability 

In [29]:
import pandas as pd

def extract_ultimate_survivors(csv_path, min_cv=0.40, min_split3=0.30, min_split1=0.30, max_dims=1500):
    df = pd.read_csv(csv_path)
    
    # Locate the exact column names dynamically
    split1_col = [c for c in df.columns if 'Split_1' in c][0]
    split3_col = [c for c in df.columns if 'Split_3' in c][0]
    
    # Apply the 4-part strict criteria mask
    mask = (
        (df['CV_Spearman'] >= min_cv) &
        (df[split3_col] >= min_split3) &
        (df[split1_col] >= min_split1) &
        (df['Num_Dims'] <= max_dims)
    )
    
    survivors = df[mask].copy()
    
    # Sort by the hardest test first (Positional), then fallback to CV score
    survivors = survivors.sort_values(by=[split3_col, 'CV_Spearman'], ascending=[False, False])
    
    print(f"\n🏆 THE ULTIMATE SURVIVORS 🏆")
    print(f"Criteria: CV >= {min_cv} | Pos (Split 3) >= {min_split3} | Dist (Split 1) >= {min_split1} | Dims <= {max_dims}\n")
    
    if survivors.empty:
        print("No models survived these strict criteria. Try relaxing the Positional threshold slightly.")
        return None
        
    print(f"Found {len(survivors)} elite architectures out of {len(df)} total models:\n")
    
    for i, row in survivors.head(15).iterrows(): # Print top 15
        print(f"Rank {row['Rank']} | Dims: {row['Num_Dims']} | Region: {row['Subregions']}")
        print(f"Features: {row['Features']}")
        print(f"   -> CV Score: {row['CV_Spearman']:.3f}")
        print(f"   -> Split 3 (Positional): {row[split3_col]:.3f}")
        print(f"   -> Split 1 (Distance):   {row[split1_col]:.3f}")
        print("-" * 75)
        
    # Save the elite list to a new CSV for final review
    output_name = csv_path.replace('.csv', '_ULTIMATE_SURVIVORS.csv')
    survivors.to_csv(output_name, index=False)
    print(f"💾 Saved complete survivor list to {output_name}")
    
    return survivors

# Execute:
# extract_ultimate_survivors('OOD_Robustness_Leaderboard_XGBoost_SUBSET_50-50_HMW%.csv')

In [62]:
extract_ultimate_survivors('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_top_performers.csv',
                           min_cv=0.50, min_split3=0.40, min_split1=0.40, max_dims=500)


🏆 THE ULTIMATE SURVIVORS 🏆
Criteria: CV >= 0.5 | Pos (Split 3) >= 0.4 | Dist (Split 1) >= 0.4 | Dims <= 500

Found 1 elite architectures out of 80 total models:

Rank 8 | Dims: 80 | Region: CD3_VH + CD3_VL
Features: Georgiev + i-AAC + Delta_G_Rank1 + 50-50_HCCF_Titer
   -> CV Score: 0.568
   -> Split 3 (Positional): 0.552
   -> Split 1 (Distance):   0.423
---------------------------------------------------------------------------
💾 Saved complete survivor list to model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_top_performers_ULTIMATE_SURVIVORS.csv


,Rank,Subregions,Features,Num_Dims,CV_Spearman,Split_1_Distance_(>15_muts)_Spearman,Split_1_Distance_(>15_muts)_HR10,Split_1_Distance_(>15_muts)_HR20,Split_1_Distance_(>15_muts)_HR30,Split_1_Distance_(>15_muts)_HR40,...,Split_2_Epistatic_(I60V_AND_V106L)_HR20,Split_2_Epistatic_(I60V_AND_V106L)_HR30,Split_2_Epistatic_(I60V_AND_V106L)_HR40,Split_2_Epistatic_(I60V_AND_V106L)_HR50,Split_3_Positional_(Pos_72)_Spearman,Split_3_Positional_(Pos_72)_HR10,Split_3_Positional_(Pos_72)_HR20,Split_3_Positional_(Pos_72)_HR30,Split_3_Positional_(Pos_72)_HR40,Split_3_Positional_(Pos_72)_HR50
7,8,CD3_VH + CD3_VL,Georgiev + i-AAC + Delta_G_Rank1 + 50-50_HCCF_...,80,0.567617,0.42305,0.25,0.4375,0.5,0.53125,...,0.625,0.75,0.75,0.761905,0.552107,0.2,0.333333,0.451613,0.595238,0.692308


In [63]:
extract_ultimate_survivors('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer_top_performers.csv',
                           min_cv=0.50, min_split3=0.45, min_split1=0.45, max_dims=2500)


🏆 THE ULTIMATE SURVIVORS 🏆
Criteria: CV >= 0.5 | Pos (Split 3) >= 0.45 | Dist (Split 1) >= 0.45 | Dims <= 2500

Found 2 elite architectures out of 103 total models:

Rank 27 | Dims: 746 | Region: CD3_VH + CD3_VL
Features: Georgiev + i-AAC + i-ESM_Small_8M + Propermab
   -> CV Score: 0.503
   -> Split 3 (Positional): 0.479
   -> Split 1 (Distance):   0.462
---------------------------------------------------------------------------
Rank 80 | Dims: 1800 | Region: CD3_VH + CD3_VL
Features: i-AAindex + i-ESM_Small_8M + Propermab
   -> CV Score: 0.503
   -> Split 3 (Positional): 0.467
   -> Split 1 (Distance):   0.458
---------------------------------------------------------------------------
💾 Saved complete survivor list to model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer_top_performers_ULTIMATE_SURVIVORS.csv


,Rank,Subregions,Features,Num_Dims,CV_Spearman,Split_1_Distance_(>15_muts)_Spearman,Split_1_Distance_(>15_muts)_HR10,Split_1_Distance_(>15_muts)_HR20,Split_1_Distance_(>15_muts)_HR30,Split_1_Distance_(>15_muts)_HR40,...,Split_2_Epistatic_(I60V_AND_V106L)_HR20,Split_2_Epistatic_(I60V_AND_V106L)_HR30,Split_2_Epistatic_(I60V_AND_V106L)_HR40,Split_2_Epistatic_(I60V_AND_V106L)_HR50,Split_3_Positional_(Pos_72)_Spearman,Split_3_Positional_(Pos_72)_HR10,Split_3_Positional_(Pos_72)_HR20,Split_3_Positional_(Pos_72)_HR30,Split_3_Positional_(Pos_72)_HR40,Split_3_Positional_(Pos_72)_HR50
26,27,CD3_VH + CD3_VL,Georgiev + i-AAC + i-ESM_Small_8M + Propermab,746,0.502900,0.461869,0.250,0.3125,0.416667,0.46875,...,0.625,0.583333,0.6250,0.666667,0.478716,0.1,0.285714,0.419355,0.595238,0.653846
79,80,CD3_VH + CD3_VL,i-AAindex + i-ESM_Small_8M + Propermab,1800,0.503286,0.457820,0.375,0.3750,0.500000,0.53125,...,0.500,0.583333,0.5625,0.571429,0.467341,0.0,0.333333,0.354839,0.547619,0.692308


In [157]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as patches

def plot_elite_survivors_dashboard(csv_path, top_n=104, test_set_sizes=None, ood_threshold=0.90, output_png="Elite_Survivors_Dashboard.png"):
    print(f"\n🌍 Loading Elite Survivors: {csv_path}")
    df = pd.read_csv(csv_path)

    # ==========================================
    # 1. Clean Up Y-Axis Labels
    # ==========================================
    def get_clean_model_name(row):
        # Strip all spaces from both regions and features immediately
        sub = str(row['Subregions']).replace("CD3_VH", "VH").replace("CD3_VL", "VL").replace("-", "+").replace(" ", "")
        feat = str(row['Features']).replace(" ", "") 
        
        feat = feat.replace("50-50_HCCF_Titer", "Titer")
        feat = feat.replace("i-", "INTERFACE_PROTECT_")
        feat = feat.replace("-", "+")
        feat = feat.replace("INTERFACE_PROTECT_", "i-")
        
        mapping = {
            "Targeted_": "i-", "Georgiev": "Geo", "AAindex": "AAidx",
            "ESM_Big_650M_SVD50": "ESM(SVD)", "ESM_Big_650M": "ESM650",
            "ESM_Medium_35M": "ESM35", "ESM_Small_8M": "ESM8",
            "AntiBERTy": "ABerty", "AbLang2_Paired": "AbL2",
            "Propermab": "pm3D", "Delta_G_Rank1": "dG", "VH_VL_Log10_Kd": "Kd"
        }
        for old, new in mapping.items():
            feat = feat.replace(old, new)
            
        short_name = f"{sub} | {feat}"
        if len(short_name) > 40:
            short_name = short_name[:37] + "..."
            
        return short_name

    df['Model_Name'] = df.apply(get_clean_model_name, axis=1)
    
    # --- STEP 1: Filter for the Elite ---
    split3_col = [c for c in df.columns if 'Split_3' in c and c.endswith('_Spearman')][0]
    df = df.sort_values(by=split3_col, ascending=False).head(top_n).copy()

    # --- STEP 2: Re-sort by Ablation Groups ---
    def get_ablation_sort_tuple(name):
        region_part, feats_part = name.split(' | ', 1) if ' | ' in name else (name, "")
        
        region_rank = 99
        if region_part == 'VH': region_rank = 1
        elif region_part == 'VL': region_rank = 2
        elif region_part == 'VH+VL': region_rank = 3
        elif 'scFv' in region_part: region_rank = 4
        
        feats = feats_part.split('+') if feats_part else []
        base_feat = feats[0].strip() if feats else ""
        
        num_feats = len(feats)
        num_targeted = sum(1 for f in feats if 'i-' in f)
        num_tabular = sum(1 for f in feats if any(k in f for k in ['dG', 'Kd', 'Titer']))
        
        return (region_rank, base_feat, num_feats, num_targeted, num_tabular, feats_part)

    df['Sort_Key'] = df['Model_Name'].apply(get_ablation_sort_tuple)
    df = df.sort_values(by='Sort_Key').drop(columns=['Sort_Key'])
    df = df.set_index('Model_Name')

    # ==========================================
    # 2. Isolate Columns & Compute Absolute N
    # ==========================================
    sp1_col = [c for c in df.columns if 'Split_1' in c and c.endswith('_Spearman')][0]
    sp2_col = [c for c in df.columns if 'Split_2' in c and c.endswith('_Spearman')][0]
    
    df_spearman = df[['CV_Spearman', sp1_col, sp2_col, split3_col]].copy()
    df_spearman.columns = ['CV', 'Dist (1)', 'Epi (2)', 'Pos (3)']

    if test_set_sizes is None:
        test_set_sizes = {'Split_1': 84, 'Split_2': 42, 'Split_3': 106}

    hr_cols = []
    hr_clean_names = []
    split_labels = [('Split_1', 'Dist'), ('Split_2', 'Epi'), ('Split_3', 'Pos')]
    
    for s_id, s_name in split_labels:
        total_test_n = test_set_sizes.get(s_id, 100)
        for k in [10, 20, 30, 40, 50]:
            col = [c for c in df.columns if s_id in c and c.endswith(f'_HR{k}')][0]
            hr_cols.append(col)
            abs_n = max(1, int(total_test_n * (k / 100.0)))
            hr_clean_names.append(f"{s_name}\n{k}%\n(N={abs_n})")
            
    df_hr = df[hr_cols].copy()
    df_hr.columns = hr_clean_names

    # ==========================================
    # 3. Plotting & Formatting
    # ==========================================
    sns.set_theme(style="white")
    
    fig, axes = plt.subplots(1, 2, figsize=(26, 12), 
                             gridspec_kw={'width_ratios': [1, 3.75], 'wspace': 0.10})
    fig.suptitle("Elite Production Architectures (OOD Validation)", fontsize=26, fontweight='bold', y=1.02)

    # --- PANEL 1: Spearman ---
    ax1 = axes[0]
    sns.heatmap(df_spearman, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax1, 
                cbar_kws={'shrink': 0.8, 'pad': 0.04, 'label': 'Spearman Correlation'},
                linewidths=1.5, linecolor='white', annot_kws={"size": 12, "weight": "bold"},
                yticklabels=True)
    
    ax1.set_title("Robustness (Spearman)", fontsize=18, fontweight='bold', pad=15)
    ax1.set_ylabel("Architecture & Features", fontsize=16, fontweight='bold')
    
    # 🌟 FIX 3: Mathematically lock the ticks to the exact center of every row
    ax1.set_yticks(np.arange(len(df_spearman)) + 0.5)
    ax1.set_yticklabels(df_spearman.index, rotation=0, fontsize=12)
    
    ax1.tick_params(axis='x', rotation=0, labelsize=14)
    # ax1.tick_params(axis='y', rotation=0, labelsize=10)
    plt.setp(ax1.get_xticklabels(), fontweight='bold')

    # Y-Axis Coloring
    unique_trails = []
    model_names = df.index.tolist()
    for name in model_names:
        region_part, feats_part = name.split(' | ', 1) if ' | ' in name else (name, "")
        base_feat = feats_part.split('+')[0].strip() if feats_part else "None"
        trail_key = f"{region_part} | {base_feat}"
        if trail_key not in unique_trails:
            unique_trails.append(trail_key)
            
    palette = sns.color_palette("dark", len(unique_trails)).as_hex()
    trail_colors = dict(zip(unique_trails, palette))
    
    for label in ax1.get_yticklabels():
        text = label.get_text()
        region_part, feats_part = text.split(' | ', 1) if ' | ' in text else (text, "")
        base_feat = feats_part.split('+')[0].strip() if feats_part else "None"
        trail_key = f"{region_part} | {base_feat}"
        label.set_color(trail_colors.get(trail_key, 'black'))
        label.set_fontweight('bold')

    # --- PANEL 2: Hit Rates ---
    ax2 = axes[1]
    sns.heatmap(df_hr, annot=True, fmt=".2f", cmap="BuPu", ax=ax2, vmin=0, vmax=1.0,
                cbar_kws={'shrink': 0.8, 'pad': 0.02, 'label': 'Hit Rate Enrichment (0.0 to 1.0)'},
                linewidths=1.5, linecolor='white', annot_kws={"size": 12, "weight": "bold"})
    
    ax2.set_title("Predictive Enrichment (Hit Rates & Absolute Matches)", fontsize=18, fontweight='bold', pad=15)
    ax2.set_ylabel("") 
    ax2.set_yticks([]) 
    ax2.tick_params(axis='x', rotation=0, labelsize=12)
    plt.setp(ax2.get_xticklabels(), fontweight='bold')
    
    # Gold vertical dividers
    ax2.axvline(x=5, color='gold', linewidth=4)
    ax2.axvline(x=10, color='gold', linewidth=4)

    # Loop through the rows to find models that beat the threshold on ALL 3 splits
    for i, (model_name, row) in enumerate(df_spearman.iterrows()):
        dist_score = row['Dist (1)']
        epi_score = row['Epi (2)']
        pos_score = row['Pos (3)']
        
        # Check if the row passes the strict OOD criteria
        if dist_score >= ood_threshold and epi_score >= ood_threshold and pos_score >= ood_threshold:
            
            # 1. Draw a thick Neon Pink / Red box around the row in Panel 1
            ax1.add_patch(patches.Rectangle(
                (0, i), len(df_spearman.columns), 1, 
                fill=False, edgecolor='#FF007F', lw=2, zorder=10, clip_on=False
            ))
            
            # 2. Draw a matching box around the row in Panel 2
            ax2.add_patch(patches.Rectangle(
                (0, i), len(df_hr.columns), 1, 
                fill=False, edgecolor='#FF007F', lw=2, zorder=10, clip_on=False
            ))
            
            # 3. Add a ⭐ to the Y-axis label to make it pop even more
            current_label = ax1.get_yticklabels()[i]
            current_text = current_label.get_text()
            current_label.set_text(f"⭐ {current_text}")
            
    # Re-apply the y-tick labels since we modified the text with stars
    ax1.set_yticklabels(ax1.get_yticklabels())
    
    plt.tight_layout()
    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"📊 Dashboard successfully saved to: {output_png}")

In [99]:
plot_elite_survivors_dashboard('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_ULTIMATE_SURVIVORS_all_champions.csv',
                              output_png="model_comparison/Elite_Survivors_Dashboard_ULTIMATE_SURVIVORS_all_champions.png")


🌍 Loading Elite Survivors: model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_ULTIMATE_SURVIVORS_all_champions.csv
📊 Dashboard successfully saved to: model_comparison/Elite_Survivors_Dashboard_ULTIMATE_SURVIVORS_all_champions.png


In [100]:
plot_elite_survivors_dashboard('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_top_performers_ULTIMATE_SURVIVORS.csv',
                              output_png="model_comparison/Elite_Survivors_Dashboard_ULTIMATE_SURVIVORS_top_performers.png")


🌍 Loading Elite Survivors: model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_top_performers_ULTIMATE_SURVIVORS.csv
📊 Dashboard successfully saved to: model_comparison/Elite_Survivors_Dashboard_ULTIMATE_SURVIVORS_top_performers.png


In [101]:
plot_elite_survivors_dashboard('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer_ULTIMATE_SURVIVORS_all_champions.csv',
                              output_png="model_comparison/Elite_Survivors_Dashboard_without_Titer_ULTIMATE_SURVIVORS_all_champions.png")


🌍 Loading Elite Survivors: model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer_ULTIMATE_SURVIVORS_all_champions.csv
📊 Dashboard successfully saved to: model_comparison/Elite_Survivors_Dashboard_without_Titer_ULTIMATE_SURVIVORS_all_champions.png


In [102]:
plot_elite_survivors_dashboard('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer_top_performers_ULTIMATE_SURVIVORS.csv',
                              output_png="model_comparison/Elite_Survivors_Dashboard_without_Titer_ULTIMATE_SURVIVORS_top_performers.png")


🌍 Loading Elite Survivors: model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer_top_performers_ULTIMATE_SURVIVORS.csv
📊 Dashboard successfully saved to: model_comparison/Elite_Survivors_Dashboard_without_Titer_ULTIMATE_SURVIVORS_top_performers.png


In [149]:
plot_elite_survivors_dashboard('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_top_performers.csv',
                               ood_threshold=0.50,
                               output_png="model_comparison/Elite_Survivors_Dashboard_top_performers.png")


🌍 Loading Elite Survivors: model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_top_performers.csv
📊 Dashboard successfully saved to: model_comparison/Elite_Survivors_Dashboard_top_performers.png


In [152]:
plot_elite_survivors_dashboard('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer_top_performers.csv',
                               ood_threshold=0.40,
                               output_png="model_comparison/Elite_Survivors_Dashboard_without_Titer_top_performers.png")


🌍 Loading Elite Survivors: model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_without_Titer_top_performers.csv
📊 Dashboard successfully saved to: model_comparison/Elite_Survivors_Dashboard_without_Titer_top_performers.png


In [159]:
plot_elite_survivors_dashboard('model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_best_holdout_models.csv',
                              output_png="model_comparison/Dashboard_best_holdout_models__.png")


🌍 Loading Elite Survivors: model_comparison/OOD_Robustness_Leaderboard_XGBoost_50-50_HMW%_best_holdout_models.csv
📊 Dashboard successfully saved to: model_comparison/Dashboard_best_holdout_models__.png
